# ResNet50 + Transfer Learning para PetFinder AdoptionSpeed

Notebook trabajado en Kaggle con GPU.

Se desarrolla un modelo de clasificación de imágenes para predecir la velocidad de adopción de mascotas utilizando redes neuronales convolucionales preentrenadas.

Se evalúan distintas configuraciones de entrenamiento considerando:
- Capas entrenadas (FC vs fine-tuning)
- Tamaño de batch
- Learning rate
- Número de epochs

El desempeño se mide mediante Quadratic Weighted Kappa (QWK).

## 1. Instalación de dependencias

Se instala Optuna para realizar la búsqueda de hiperparámetros.  
El resto de las librerías utilizadas ya se encuentran disponibles en el entorno de Kaggle.

In [2]:
!pip install optuna -q

## 2. Imports

In [12]:
import os
import shutil
import time
import copy
import datetime
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, cohen_kappa_score

import optuna

import torch
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

from joblib import dump, load

## 3. Configuración del entorno de cómputo

Se verifica la disponibilidad de GPU en Kaggle y se define el dispositivo de entrenamiento.

In [13]:
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CUDA disponible: True
GPU: Tesla T4
Device: cuda


## 4. Rutas del dataset en Kaggle

El dataset fue agregado al notebook desde la competencia **PetFinder Adoption Prediction**.  
En Kaggle, los datos de entrada se encuentran en `/kaggle/input`, mientras que los archivos generados durante la ejecución se guardan en `/kaggle/working`.

In [14]:
INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

print("Contenido de /kaggle/input:")
print(os.listdir(INPUT_DIR))

print("Contenido de /kaggle/working:")
print(os.listdir(WORKING_DIR))

Contenido de /kaggle/input:
['competitions']
Contenido de /kaggle/working:
['.virtual_documents']


In [15]:
DATA_DIR = "/kaggle/input/competitions/petfinder-adoption-prediction"
WORK_DIR = "/kaggle/working"

TRAIN_CSV = f"{DATA_DIR}/train/train.csv"
TEST_CSV = f"{DATA_DIR}/test/test.csv"

TRAIN_IMG_DIR = f"{DATA_DIR}/train_images"
TEST_IMG_DIR = f"{DATA_DIR}/test_images"

TRAIN_META_DIR = f"{DATA_DIR}/train_metadata"
TEST_META_DIR = f"{DATA_DIR}/test_metadata"

TRAIN_SENTIMENT_DIR = f"{DATA_DIR}/train_sentiment"
TEST_SENTIMENT_DIR = f"{DATA_DIR}/test_sentiment"

BREED_LABELS = f"{DATA_DIR}/breed_labels.csv"
COLOR_LABELS = f"{DATA_DIR}/color_labels.csv"
STATE_LABELS = f"{DATA_DIR}/state_labels.csv"

## 5. Carga inicial del archivo train.csv

Se carga la tabla principal de entrenamiento.  
Cada fila representa una mascota e incluye variables descriptivas, el identificador `PetID` y la variable objetivo `AdoptionSpeed`.

In [16]:
df_train = pd.read_csv(TRAIN_CSV)

print(df_train.shape)
df_train.head()

(14993, 24)


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,2,Nibble,3,299,0,1,1,7,0,1,...,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0,2
1,2,No Name Yet,1,265,0,1,1,2,0,2,...,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.0,0
2,1,Brisco,1,307,0,1,2,7,0,2,...,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.0,3
3,1,Miko,4,307,0,2,1,2,0,2,...,1,1,150,41401,9238e4f44c71a75282e62f7136c6b240,0,"Good guard dog, very alert, active, obedience ...",5842f1ff5,8.0,2
4,1,Hunter,1,307,0,1,1,0,0,2,...,1,1,0,41326,95481e953f8aed9ec3d16fc4509537e8,0,This handsome yet cute boy is up for adoption....,850a43f90,3.0,2


## 6. Paths y parámetros generales

Se definen las rutas de trabajo, carpetas para guardar modelos, matrices de confusión y resultados de Optuna.  
También se fijan parámetros generales del entrenamiento.

In [17]:
DATA_DIR = "/kaggle/input/competitions/petfinder-adoption-prediction"
WORK_DIR = "/kaggle/working"

PATH_TO_TRAIN = f"{DATA_DIR}/train/train.csv"
PATH_TO_IMAGES_DIR = f"{DATA_DIR}/train_images"

PATH_TO_WORK = WORK_DIR
PATH_TO_TEMP_FILES = os.path.join(PATH_TO_WORK, "optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(PATH_TO_WORK, "optuna_artifacts")
PATH_TO_MODELS = os.path.join(PATH_TO_WORK, "models")
PATH_TO_CMS = os.path.join(PATH_TO_WORK, "confusion_matrices")

# Crear carpetas
for path in [PATH_TO_WORK, PATH_TO_TEMP_FILES, PATH_TO_OPTUNA_ARTIFACTS, PATH_TO_MODELS, PATH_TO_CMS]:
    os.makedirs(path, exist_ok=True)

# ======================
# OPTUNA (KAGGLE)
# ======================

OPTUNA_DB = f"{WORK_DIR}/optuna.db"

# ======================
# CONFIG
# ======================

MODEL_NAME = "ResNet50"
MODEL_VERSION = "1.0.0"

SEED = 42
BATCH_SIZE = 32
TEST_SIZE = 0.2
IMAGE_SIZE = 224
NUM_WORKERS = 2

class_names = ['0', '1', '2', '3', '4']
class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}

# ======================
# SEED + DEVICE
# ======================

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 7. Lectura del CSV y verificación de imágenes

Se carga nuevamente el archivo `train.csv` usando la ruta definitiva del proyecto.  
Además, se verifica que las imágenes estén disponibles en el directorio esperado.

In [18]:
train_df = pd.read_csv(PATH_TO_TRAIN)

print("Shape train.csv:", train_df.shape)
display(train_df.head())

print("Cantidad de archivos en train_images:", len(os.listdir(PATH_TO_IMAGES_DIR)))
print("Primeros archivos:", os.listdir(PATH_TO_IMAGES_DIR)[:5])

print("\nDistribución AdoptionSpeed:")
display(
    (train_df["AdoptionSpeed"]
     .value_counts(normalize=True)
     .sort_index()
     .mul(100)
     .round(2)
     .to_frame("porcentaje"))
)

Shape train.csv: (14993, 24)


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,2,Nibble,3,299,0,1,1,7,0,1,...,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0,2
1,2,No Name Yet,1,265,0,1,1,2,0,2,...,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.0,0
2,1,Brisco,1,307,0,1,2,7,0,2,...,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.0,3
3,1,Miko,4,307,0,2,1,2,0,2,...,1,1,150,41401,9238e4f44c71a75282e62f7136c6b240,0,"Good guard dog, very alert, active, obedience ...",5842f1ff5,8.0,2
4,1,Hunter,1,307,0,1,1,0,0,2,...,1,1,0,41326,95481e953f8aed9ec3d16fc4509537e8,0,This handsome yet cute boy is up for adoption....,850a43f90,3.0,2


Cantidad de archivos en train_images: 58311
Primeros archivos: ['cf8d949f9-2.jpg', '61d4dc56b-12.jpg', '53923463d-9.jpg', '0173c456c-8.jpg', 'fa7c7d1be-3.jpg']

Distribución AdoptionSpeed:


,porcentaje
AdoptionSpeed,
0,2.73
1,20.61
2,26.93
3,21.74
4,27.99


## 8. Funciones de visualización

Se definen funciones auxiliares para inspeccionar imágenes del dataset.

In [20]:
def visualize_pet(pet_id):
    path_to_image = os.path.join(PATH_TO_IMAGES_DIR, f'{pet_id}-1.jpg')
    image_to_show = cv2.imread(path_to_image)
    if image_to_show is None:
        print(f"No se encontró imagen para PetID={pet_id}")
        return
    image_to_show = cv2.cvtColor(image_to_show, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(5, 5))
    plt.imshow(image_to_show)
    plt.axis('off')
    plt.show()

def visualize_image(image):
    image = cv2.convertScaleAbs(image)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(5, 5))
    plt.imshow(image.astype(np.uint8))
    plt.axis('off')
    plt.show()

## 9. Split estratificado y organización de imágenes

Para entrenar el modelo con `ImageFolder`, las imágenes se copian a carpetas organizadas por clase.

Se realiza una división estratificada entre entrenamiento y validación para preservar la distribución de `AdoptionSpeed` en ambos subconjuntos.

In [21]:
CREATE_PYTORCH_DIRECTORIES = 1

new_train_directory = os.path.join(PATH_TO_WORK, "train_images_classes")
new_val_directory = os.path.join(PATH_TO_WORK, "val_images_classes")

os.makedirs(new_train_directory, exist_ok=True)
os.makedirs(new_val_directory, exist_ok=True)

for clase in class_names:
    os.makedirs(os.path.join(new_train_directory, clase), exist_ok=True)
    os.makedirs(os.path.join(new_val_directory, clase), exist_ok=True)

train_data, val_data = train_test_split(
    train_df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=train_df["AdoptionSpeed"]
)

train_data = train_data.copy()
val_data = val_data.copy()

print("Train:", train_data.shape)
print("Val:", val_data.shape)

print("\nDistribución train (%):")
print(
    train_data["AdoptionSpeed"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nDistribución val (%):")
print(
    val_data["AdoptionSpeed"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Train: (11994, 24)
Val: (2999, 24)

Distribución train (%):
AdoptionSpeed
0     2.73
1    20.61
2    26.93
3    21.74
4    27.99
Name: proportion, dtype: float64

Distribución val (%):
AdoptionSpeed
0     2.73
1    20.61
2    26.91
3    21.74
4    28.01
Name: proportion, dtype: float64


## 10. Copia de imágenes a carpetas por clase

Se copia la primera imagen disponible de cada mascota (`PetID-1.jpg`) al directorio correspondiente según su clase de `AdoptionSpeed`.

Este paso permite usar `datasets.ImageFolder`, que espera una estructura de carpetas donde cada subcarpeta representa una clase.

In [22]:
def copy_images_to_class_dirs(data, destination_dir):
    missing = 0
    copied = 0

    for _, row in tqdm(data.iterrows(), total=len(data)):
        pet_id = row["PetID"]
        adoption_speed = row["AdoptionSpeed"]

        file_name = f"{pet_id}-1.jpg"
        source = os.path.join(PATH_TO_IMAGES_DIR, file_name)
        target = os.path.join(destination_dir, str(adoption_speed), file_name)

        if os.path.exists(source):
            if not os.path.exists(target):
                shutil.copy2(source, target)
                copied += 1
        else:
            missing += 1

    print(f"Destino: {destination_dir}")
    print(f"Copiadas nuevas: {copied}")
    print(f"Faltantes: {missing}")

if CREATE_PYTORCH_DIRECTORIES == 1:
    copy_images_to_class_dirs(train_data, new_train_directory)
    copy_images_to_class_dirs(val_data, new_val_directory)
    print("Proceso de copia completado.")
else:
    print("Se omite copia de imágenes.")

100%|██████████| 11994/11994 [02:14<00:00, 89.29it/s] 


Destino: /kaggle/working/train_images_classes
Copiadas nuevas: 11721
Faltantes: 273


100%|██████████| 2999/2999 [00:29<00:00, 101.88it/s]

Destino: /kaggle/working/val_images_classes
Copiadas nuevas: 2931
Faltantes: 68
Proceso de copia completado.


In [23]:
for clase in class_names:
    train_count = len(os.listdir(os.path.join(new_train_directory, clase)))
    val_count = len(os.listdir(os.path.join(new_val_directory, clase)))
    print(f"Clase {clase} | Train: {train_count} | Val: {val_count}")

Clase 0 | Train: 307 | Val: 79
Clase 1 | Train: 2445 | Val: 611
Clase 2 | Train: 3203 | Val: 798
Clase 3 | Train: 2584 | Val: 641
Clase 4 | Train: 3182 | Val: 802


Luego de la copia, se verifica que existan imágenes en cada carpeta de clase. Este control es necesario porque `ImageFolder` fallará si las carpetas están vacías.

## 11. Data augmentation y DataLoaders

Para entrenamiento se aplican transformaciones moderadas con el objetivo de mejorar la generalización del modelo.

Para validación no se utiliza data augmentation; solo se aplican transformaciones determinísticas.

In [24]:
def create_dataloaders(train_directory, val_directory, batch_size, num_workers):
    train_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    train_dataset = datasets.ImageFolder(train_directory, transform=train_transforms)
    val_dataset = datasets.ImageFolder(val_directory, transform=val_transforms)

    train_dataset.class_to_idx = class_to_idx
    val_dataset.class_to_idx = class_to_idx

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    return train_dataset, val_dataset, train_loader, val_loader

train_dataset, val_dataset, train_dataloader, val_dataloader = create_dataloaders(
    new_train_directory,
    new_val_directory,
    BATCH_SIZE,
    NUM_WORKERS
)

print("Train images cargadas:", len(train_dataset))
print("Val images cargadas:", len(val_dataset))
print("Clases:", train_dataset.classes)

Train images cargadas: 11721
Val images cargadas: 2931
Clases: ['0', '1', '2', '3', '4']


Las clases son detectadas automáticamente por `ImageFolder` a partir del nombre de las carpetas. En este caso, las clases corresponden a los valores de `AdoptionSpeed`.

## 12. Función para matriz de confusión

Se define una función auxiliar para visualizar matrices de confusión.  
Esta matriz permite identificar qué clases son confundidas con mayor frecuencia por el modelo.

In [25]:
def plot_confusion_matrix(y_true, y_pred, classes=None, normalize=False, title="Matriz de confusión"):
    cm = confusion_matrix(y_true, y_pred)

    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm = np.nan_to_num(cm)

    if classes is None:
        classes = [str(i) for i in range(cm.shape[0])]

    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(cm)
    fig.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(len(classes)),
        yticks=np.arange(len(classes)),
        xticklabels=classes,
        yticklabels=classes,
        ylabel="True label",
        xlabel="Predicted label",
        title=title
    )

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = f"{cm[i, j]:.2f}" if normalize else f"{int(cm[i, j])}"
            ax.text(
                j, i, value,
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    fig.tight_layout()
    return fig

## 13. Función de entrenamiento y validación

La función `train_val` entrena el modelo y evalúa su desempeño en validación en cada epoch.

Para cada fase calcula:
- loss;
- accuracy;
- Quadratic Weighted Kappa.

El mejor modelo se selecciona según el valor de QWK en validación.

In [26]:
def train_val(
    model,
    criterion,
    dataloaders,
    datasets,
    device,
    num_epochs=5,
    lr=0.001,
    momentum=0.9,
    experiment_name="experiment"
):
    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        momentum=momentum
    )

    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_kappa = -999
    best_cm_path = None

    history = []

    for epoch in range(num_epochs):
        print(f"Epoch {epoch}/{num_epochs - 1}")
        print("-" * 10)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            y_true_epoch = []
            y_pred_epoch = []

            for inputs, labels in tqdm(dataloaders[phase]):
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

                if phase == "val":
                    y_true_epoch.extend(labels.detach().cpu().numpy().tolist())
                    y_pred_epoch.extend(preds.detach().cpu().numpy().tolist())

            epoch_loss = running_loss / len(datasets[phase])
            epoch_acc = running_corrects.double().item() / len(datasets[phase])

            if phase == "val":
                epoch_kappa = cohen_kappa_score(
                    y_true_epoch,
                    y_pred_epoch,
                    weights="quadratic"
                )
            else:
                epoch_kappa = np.nan

            print(
                f"{phase.title()} Loss: {epoch_loss:.4f} "
                f"Acc: {epoch_acc * 100:.2f}% "
                f"Kappa: {epoch_kappa if not np.isnan(epoch_kappa) else np.nan:.3f}"
            )

            history.append({
                "experiment": experiment_name,
                "epoch": epoch,
                "phase": phase,
                "loss": epoch_loss,
                "accuracy": epoch_acc,
                "kappa": epoch_kappa
            })

            if phase == "val" and epoch_kappa > best_kappa:
                best_acc = epoch_acc
                best_kappa = epoch_kappa
                best_model_wts = copy.deepcopy(model.state_dict())

                cm_filename = f"{experiment_name}_epoch_{epoch}_kappa_{epoch_kappa:.3f}.jpg"
                cm_path = os.path.join(PATH_TO_CMS, cm_filename)

                fig = plot_confusion_matrix(
                    y_true_epoch,
                    y_pred_epoch,
                    classes=class_names,
                    normalize=False,
                    title=f"{experiment_name} - CM epoch {epoch}"
                )
                fig.savefig(cm_path, dpi=150, bbox_inches="tight")
                plt.close(fig)
                best_cm_path = cm_path

        print()

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best val Acc: {best_acc * 100:.2f}%")
    print(f"Best val Kappa: {best_kappa:.3f}")
    if best_cm_path:
        print(f"Best confusion matrix saved at: {best_cm_path}")

    model.load_state_dict(best_model_wts)

    run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    model_path = os.path.join(PATH_TO_MODELS, f"{experiment_name}_{run_id}.pth")
    torch.save(model.state_dict(), model_path)
    print(f"Modelo guardado en: {model_path}")

    return model, best_kappa, pd.DataFrame(history), model_path, best_cm_path

## 14. Construcción del modelo ResNet50

Se utiliza ResNet50 preentrenada y se reemplaza la capa final para adaptar la salida a cinco clases.

Según el experimento, se entrenan:
- solo la capa fully connected final;
- la capa final + `layer4`.

Esto permite comparar una estrategia de extracción de características frente a una estrategia de fine-tuning.

In [27]:
def build_resnet50_model(
    num_classes=5,
    unfreeze_layer4=False,
    unfreeze_layer3=False
):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    if unfreeze_layer4:
        for param in model.layer4.parameters():
            param.requires_grad = True

    if unfreeze_layer3:
        for param in model.layer3.parameters():
            param.requires_grad = True

    model = model.to(device)
    return model


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

## 15. Guardado de resultados

Se define una función auxiliar para guardar modelos, matrices de confusión e historiales de entrenamiento.

In [28]:
SAVE_DIR = "/kaggle/working/experimentos"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(f"{SAVE_DIR}/models", exist_ok=True)
os.makedirs(f"{SAVE_DIR}/confusion_matrices", exist_ok=True)
os.makedirs(f"{SAVE_DIR}/histories", exist_ok=True)
os.makedirs(f"{SAVE_DIR}/studies", exist_ok=True)

def save_experiment_outputs(exp_name, model_path, cm_path, history=None, study=None):
    if model_path is not None and os.path.exists(model_path):
        shutil.copy(model_path, f"{SAVE_DIR}/models/{exp_name}_best_model.pth")

    if cm_path is not None and os.path.exists(cm_path):
        shutil.copy(cm_path, f"{SAVE_DIR}/confusion_matrices/{exp_name}_confusion_matrix.jpg")

    if history is not None:
        joblib.dump(history, f"{SAVE_DIR}/histories/{exp_name}_history.pkl")

    if study is not None:
        joblib.dump(study, f"{SAVE_DIR}/studies/{exp_name}_study.pkl")

    pd.DataFrame(results).to_csv(f"{SAVE_DIR}/results_summary.csv", index=False)

    print(f"Guardado completo en: {SAVE_DIR}")

## 16. Inicialización de contenedores de resultados

Se inicializan listas para guardar los resultados resumidos y los historiales de entrenamiento de todos los experimentos.

In [ ]:
results = []
all_histories = []

# Experimentos

A partir de la arquitectura ResNet50 preentrenada, se diseñaron distintos experimentos para evaluar el impacto de tres decisiones principales:

1. entrenar solo la capa final fully connected o realizar fine-tuning de `layer4`;
2. modificar el tamaño de batch;
3. optimizar hiperparámetros mediante Optuna.

En todos los casos, el desempeño se evalúa sobre el conjunto de validación mediante Quadratic Weighted Kappa (QWK).

### Experimento 1 — Optimización de la capa final fully connected

En este primer experimento se entrena únicamente la capa final del modelo (`fc`), manteniendo congeladas las capas convolucionales de ResNet50.

Este experimento funciona como línea base, ya que permite evaluar cuánto desempeño se obtiene utilizando la red preentrenada como extractor fijo de características.

In [27]:
EXP_NAME = "exp1_optuna_fc_only"

def optuna_train_exp1(trial):
    print(f"\nTRIAL {trial.number}")

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    momentum = trial.suggest_float("momentum", 0.80, 0.95)
    epochs = trial.suggest_int("epochs", 4, 6)

    model = build_resnet50_model(
        num_classes=5,
        unfreeze_layer4=False,
        unfreeze_layer3=False
    )

    criterion = nn.CrossEntropyLoss()

    model, kappa, history, model_path, cm_path = train_val(
        model,
        criterion,
        {"train": train_dataloader, "val": val_dataloader},
        {"train": train_dataset, "val": val_dataset},
        device,
        epochs,
        lr,
        momentum,
        f"{EXP_NAME}_trial_{trial.number}"
    )

    results.append({
        "experiment": EXP_NAME,
        "trial": trial.number,
        "batch_size": BATCH_SIZE,
        "unfreeze_layer4": False,
        "unfreeze_layer3": False,
        "lr": lr,
        "momentum": momentum,
        "epochs": epochs,
        "kappa": kappa,
        "model_path": model_path,
        "cm_path": cm_path
    })

    all_histories.append(history)

    trial.set_user_attr("model_path", model_path)
    trial.set_user_attr("cm_path", cm_path)

    return kappa


study_exp1 = optuna.create_study(
    direction="maximize",
    study_name=EXP_NAME,
    storage=f"sqlite:///{OPTUNA_DB}",
    load_if_exists=True
)

study_exp1.optimize(optuna_train_exp1, n_trials=20)

pd.DataFrame(results).to_csv(f"/kaggle/working/{EXP_NAME}_results.csv", index=False)
shutil.copy(OPTUNA_DB, f"/kaggle/working/{EXP_NAME}_optuna_backup.db")

print("Experimento 1 finalizado y respaldado.")

[I 2026-04-29 11:50:50,662] Using an existing study with name 'exp1_optuna_fc_only' instead of creating a new one.



TRIAL 1
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 187MB/s]


Epoch 0/4
----------


100%|██████████| 367/367 [00:43<00:00,  8.46it/s]


...

100%|██████████| 367/367 [00:45<00:00,  8.14it/s]


Train Loss: 1.3477 Acc: 39.39% Kappa: nan


100%|██████████| 92/92 [00:10<00:00,  8.59it/s]
[I 2026-04-29 13:20:40,480] Trial 20 finished with value: 0.2827752606647319 and parameters: {'lr': 0.00956319538241362, 'momentum': 0.8048399004601832, 'epochs': 5}. Best is trial 13 with value: 0.2965774006415449.


Val Loss: 1.4486 Acc: 31.18% Kappa: 0.217

Training complete in 4m 39s
Best val Acc: 35.28%
Best val Kappa: 0.283
Best confusion matrix saved at: /kaggle/working/confusion_matrices/exp1_optuna_fc_only_trial_20_epoch_1_kappa_0.283.jpg
Modelo guardado en: /kaggle/working/models/exp1_optuna_fc_only_trial_20_20260429_132040.pth


'/kaggle/working/exp1_optuna_fc_only_optuna_backup_part1.db'

In [ ]:
best_exp1 = study_exp1.best_trial

print("Mejor trial exp1:", best_exp1.number)
print("Mejor QWK exp1:", best_exp1.value)
print("Mejores hiperparámetros exp1:", best_exp1.params)

### Experimento 2 — Fine-tuning de layer4 (FC + layer4, batch = 32)

En este experimento se realiza fine-tuning parcial del modelo, entrenando:

- la capa fully connected final (`fc`);
- la última capa convolucional (`layer4`) de ResNet50.

Se mantiene el tamaño de batch en 32 y se optimizan mediante Optuna:

- learning rate (en un rango más bajo);
- momentum;
- número de epochs.

Se utiliza un rango de learning rate más pequeño que en el Experimento 1 por el entrenamiento de capas más profundas.

In [29]:
EXP_NAME = "exp2_optuna_fc_layer4"

def optuna_train_exp2(trial):
    print(f"\nTRIAL {trial.number}")

    lr = trial.suggest_float("lr", 1e-6, 1e-4, log=True)
    momentum = trial.suggest_float("momentum", 0.85, 0.98)
    epochs = trial.suggest_int("epochs", 4, 6)

    model = build_resnet50_model(
        num_classes=5,
        unfreeze_layer4=True,
        unfreeze_layer3=False
    )

    criterion = nn.CrossEntropyLoss()

    model, kappa, history, model_path, cm_path = train_val(
        model,
        criterion,
        {"train": train_dataloader, "val": val_dataloader},
        {"train": train_dataset, "val": val_dataset},
        device,
        epochs,
        lr,
        momentum,
        f"{EXP_NAME}_trial_{trial.number}"
    )

    results.append({
        "experiment": EXP_NAME,
        "trial": trial.number,
        "lr": lr,
        "momentum": momentum,
        "epochs": epochs,
        "kappa": kappa,
        "model_path": model_path,
        "cm_path": cm_path
    })

    all_histories.append(history)

    trial.set_user_attr("model_path", model_path)
    trial.set_user_attr("cm_path", cm_path)

    return kappa


study_exp2 = optuna.create_study(
    direction="maximize",
    study_name=EXP_NAME,
    storage=f"sqlite:///{OPTUNA_DB}",
    load_if_exists=True
)

study_exp2.optimize(optuna_train_exp2, n_trials=20)

pd.DataFrame(results).to_csv(f"/kaggle/working/{EXP_NAME}_results.csv", index=False)
shutil.copy(OPTUNA_DB, f"/kaggle/working/{EXP_NAME}_optuna_backup.db")

print("Experimento 2 finalizado y respaldado.")

[I 2026-04-29 14:33:31,895] Using an existing study with name 'exp2_optuna_fc_layer4' instead of creating a new one.



TRIAL 13
Epoch 0/3
----------


100%|██████████| 367/367 [00:59<00:00,  6.12it/s]


Train Loss: 1.4705 Acc: 31.05% Kappa: nan


100%|██████████| 92/92 [00:10<00:00,  8.72it/s]


...

Train Loss: 1.4932 Acc: 29.85% Kappa: nan


100%|██████████| 92/92 [00:10<00:00,  8.77it/s]


Val Loss: 1.4380 Acc: 33.13% Kappa: 0.196

Epoch 1/3
----------


 41%|████      | 150/367 [00:24<00:35,  6.05it/s]
[W 2026-04-29 15:34:13,543] Trial 25 failed with parameters: {'lr': 4.220207443000859e-05, 'momentum': 0.9798144646230492, 'epochs': 4} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_57/234170156.py", line 18, in optuna_train_exp2
    model, kappa, history, model_path, cm_path = train_val(
                                                 ^^^^^^^^^^
  File "/tmp/ipykernel_57/1710094246.py", line 58, in train_val
    running_loss += loss.item() * inputs.size(0)
                    ^^^^^^^^^^^
KeyboardInterrupt
[W 2026-04-29 15:34:13,545] Trial 25 failed with value None.


KeyboardInterrupt: 

In [30]:
print("Cantidad de trials:", len(study_exp2.trials))
print("Mejor trial:", study_exp2.best_trial)

Cantidad de trials: 26
Mejor trial: FrozenTrial(number=10, state=<TrialState.COMPLETE: 1>, values=[0.29111580868405584], datetime_start=datetime.datetime(2026, 4, 29, 14, 22, 3, 269096), datetime_complete=datetime.datetime(2026, 4, 29, 14, 26, 46, 455024), params={'lr': 9.854087684449751e-05, 'momentum': 0.9789049342924199, 'epochs': 4}, user_attrs={'cm_path': '/kaggle/working/confusion_matrices/exp2_optuna_fc_layer4_trial_10_epoch_3_kappa_0.291.jpg', 'model_path': '/kaggle/working/models/exp2_optuna_fc_layer4_trial_10_20260429_142646.pth'}, system_attrs={}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.0001, log=True, low=1e-06, step=None), 'momentum': FloatDistribution(high=0.98, log=False, low=0.85, step=None), 'epochs': IntDistribution(high=6, log=False, low=4, step=1)}, trial_id=32, value=None)


El mejor desempeño obtenido fue:

- **QWK ≈ 0.291**
- learning rate ≈ 9.85e-05  
- momentum ≈ 0.979  
- epochs = 4

### Experimento 3 — Fine-tuning de layer4 con batch size 16

En este experimento se mantiene la estrategia de fine-tuning parcial del Experimento 2, entrenando la capa final (`fc`) y `layer4`.

La diferencia principal es que se reduce el tamaño de batch de 32 a 16.

Esta modificación busca evaluar si un batch más pequeño mejora la generalización del modelo. 

Se optimizan mediante Optuna:

- learning rate;
- momentum;
- número de epochs.

In [31]:
EXP_NAME = "exp3_optuna_fc_layer4_batch16"

BATCH_SIZE_EXP3 = 16
train_ds3, val_ds3, train_dl3, val_dl3 = create_dataloaders(
    new_train_directory,
    new_val_directory,
    BATCH_SIZE_EXP3,
    NUM_WORKERS
)

def optuna_train_exp3(trial):
    print(f"\nTRIAL {trial.number}")

    lr = trial.suggest_float("lr", 1e-5, 5e-4, log=True)
    momentum = trial.suggest_float("momentum", 0.80, 0.95)
    epochs = trial.suggest_int("epochs", 5, 8)

    model = build_resnet50_model(5, True, False)

    criterion = nn.CrossEntropyLoss()

    model, kappa, history, model_path, cm_path = train_val(
        model, criterion,
        {"train": train_dl3, "val": val_dl3},
        {"train": train_data, "val": val_data},
        device, epochs, lr, momentum,
        f"{EXP_NAME}_trial_{trial.number}"
    )

    trial.set_user_attr("model_path", model_path)
    trial.set_user_attr("cm_path", cm_path)

    return kappa


study_exp3 = optuna.create_study(
    direction="maximize",
    study_name=EXP_NAME,
    storage=f"sqlite:///{OPTUNA_DB}",
    load_if_exists=True
)

study_exp3.optimize(optuna_train_exp3, n_trials=20)

pd.DataFrame(results).to_csv(f"/kaggle/working/{EXP_NAME}_results.csv", index=False)
shutil.copy(OPTUNA_DB, f"/kaggle/working/{EXP_NAME}_optuna_backup.db")

print("Experimento 3 finalizado y respaldado.")

[I 2026-04-29 15:34:42,689] A new study created in RDB with name: exp3_optuna_fc_layer4_batch16



TRIAL 0
Epoch 0/6
----------


100%|██████████| 733/733 [01:04<00:00, 11.43it/s]


Train Loss: 1.4994 Acc: 26.95% Kappa: nan


100%|██████████| 184/184 [00:10<00:00, 17.97it/s]


...

100%|██████████| 733/733 [01:01<00:00, 11.92it/s]


Train Loss: 1.3176 Acc: 38.79% Kappa: nan


100%|██████████| 184/184 [00:10<00:00, 17.29it/s]


Val Loss: 1.3634 Acc: 35.38% Kappa: 0.289

Training complete in 9m 41s
Best val Acc: 36.11%
Best val Kappa: 0.305
Best confusion matrix saved at: /kaggle/working/confusion_matrices/exp3_optuna_fc_layer4_batch16_trial_19_epoch_6_kappa_0.305.jpg
Modelo guardado en: /kaggle/working/models/exp3_optuna_fc_layer4_batch16_trial_19_20260429_182453.pth


[I 2026-04-29 18:24:53,371] Trial 19 finished with value: 0.3046099070150169 and parameters: {'lr': 0.00045672184599701186, 'momentum': 0.8335942872210407, 'epochs': 8}. Best is trial 16 with value: 0.3117360983701776.


In [34]:
print("Cantidad de trials:", len(study_exp3.trials))
print("Mejor trial:", study_exp3.best_trial)

Cantidad de trials: 20
Mejor trial: FrozenTrial(number=16, state=<TrialState.COMPLETE: 1>, values=[0.3117360983701776], datetime_start=datetime.datetime(2026, 4, 29, 17, 47, 14, 572869), datetime_complete=datetime.datetime(2026, 4, 29, 17, 56, 57, 48824), params={'lr': 0.00041194164591519024, 'momentum': 0.8311181411720225, 'epochs': 8}, user_attrs={'cm_path': '/kaggle/working/confusion_matrices/exp3_optuna_fc_layer4_batch16_trial_16_epoch_4_kappa_0.312.jpg', 'model_path': '/kaggle/working/models/exp3_optuna_fc_layer4_batch16_trial_16_20260429_175656.pth'}, system_attrs={}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.0005, log=True, low=1e-05, step=None), 'momentum': FloatDistribution(high=0.95, log=False, low=0.8, step=None), 'epochs': IntDistribution(high=8, log=False, low=5, step=1)}, trial_id=64, value=None)


### Resultados del Experimento 3

El mejor desempeño se obtuvo en el trial 16:

- **QWK ≈ 0.312**
- learning rate ≈ 4.12e-4
- momentum ≈ 0.831
- epochs = 8
- batch size = 16

Este fue el mejor resultado obtenido durante la optimización.

### Experimento 4 — Fine-tuning de layer4 con learning rate fino

En este experimento se mantiene la configuración del Experimento 3:

- entrenamiento de la capa final (`fc`);
- fine-tuning de `layer4`;
- batch size = 16.

La diferencia principal es que se restringe el rango del learning rate a valores más bajos. Esta estrategia busca evaluar si un ajuste más conservador de los pesos preentrenados mejora la estabilidad del entrenamiento.

Se optimizan mediante Optuna:

- learning rate en un rango fino;
- momentum;
- número de epochs.

In [29]:
EXP_NAME = "exp4_optuna_fc_layer4_batch16"

BATCH_SIZE_EXP4 = 16

train_ds4, val_ds4, train_dl4, val_dl4 = create_dataloaders(
    new_train_directory,
    new_val_directory,
    BATCH_SIZE_EXP4,
    NUM_WORKERS
)

def optuna_train_exp4(trial):
    print(f"\nTRIAL {trial.number}")

    lr = trial.suggest_float("lr", 5e-6, 2e-4, log=True)
    momentum = trial.suggest_float("momentum", 0.88, 0.98)
    epochs = trial.suggest_int("epochs", 5, 8)

    model = build_resnet50_model(
        num_classes=5,
        unfreeze_layer4=True,
        unfreeze_layer3=False
    )

    criterion = nn.CrossEntropyLoss()

    model, kappa, history, model_path, cm_path = train_val(
        model,
        criterion,
        {"train": train_dl4, "val": val_dl4},
        {"train": train_ds4, "val": val_ds4},
        device,
        epochs,
        lr,
        momentum,
        f"{EXP_NAME}_trial_{trial.number}"
    )

    results.append({
        "experiment": EXP_NAME,
        "trial": trial.number,
        "batch_size": BATCH_SIZE_EXP4,
        "lr": lr,
        "momentum": momentum,
        "epochs": epochs,
        "kappa": kappa,
        "model_path": model_path,
        "cm_path": cm_path
    })

    all_histories.append(history)

    trial.set_user_attr("model_path", model_path)
    trial.set_user_attr("cm_path", cm_path)

    return kappa


study_exp4 = optuna.create_study(
    direction="maximize",
    study_name=EXP_NAME,
    storage=f"sqlite:///{OPTUNA_DB}",
    load_if_exists=True
)

study_exp4.optimize(optuna_train_exp4, n_trials=20)

pd.DataFrame(results).to_csv(f"/kaggle/working/{EXP_NAME}_results.csv", index=False)
shutil.copy(OPTUNA_DB, f"/kaggle/working/{EXP_NAME}_optuna_backup.db")

print("Experimento 4 finalizado y respaldado.")

[I 2026-04-30 11:20:34,258] A new study created in RDB with name: exp4_optuna_fc_layer4_batch16



TRIAL 0
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 179MB/s] 


Epoch 0/4
----------


100%|██████████| 733/733 [00:50<00:00, 14.42it/s]


...

Val Loss: 1.3971 Acc: 35.89% Kappa: 0.288

Epoch 5/5
----------


100%|██████████| 733/733 [00:53<00:00, 13.79it/s]


Train Loss: 1.3791 Acc: 37.64% Kappa: nan


100%|██████████| 184/184 [00:09<00:00, 19.88it/s]
[I 2026-04-30 13:45:43,750] Trial 19 finished with value: 0.287681579625781 and parameters: {'lr': 0.00011181264366980306, 'momentum': 0.9436297548331631, 'epochs': 6}. Best is trial 6 with value: 0.30852112095218387.


Val Loss: 1.3954 Acc: 36.03% Kappa: 0.285

Training complete in 6m 16s
Best val Acc: 35.89%
Best val Kappa: 0.288
Best confusion matrix saved at: /kaggle/working/confusion_matrices/exp4_optuna_fc_layer4_batch16_trial_19_epoch_4_kappa_0.288.jpg
Modelo guardado en: /kaggle/working/models/exp4_optuna_fc_layer4_batch16_trial_19_20260430_134543.pth
Todo guardado ✔


In [56]:
print("Cantidad de trials:", len(study_exp4.trials))
print("Mejor trial:", study_exp4.best_trial)

Cantidad de trials: 20
Mejor trial: FrozenTrial(number=6, state=<TrialState.COMPLETE: 1>, values=[0.30852112095218387], datetime_start=datetime.datetime(2026, 4, 30, 12, 3, 21, 328225), datetime_complete=datetime.datetime(2026, 4, 30, 12, 10, 38, 929494), params={'lr': 0.00014184777695228404, 'momentum': 0.9773997523560716, 'epochs': 7}, user_attrs={'cm_path': '/kaggle/working/confusion_matrices/exp4_optuna_fc_layer4_batch16_trial_6_epoch_2_kappa_0.309.jpg', 'model_path': '/kaggle/working/models/exp4_optuna_fc_layer4_batch16_trial_6_20260430_121038.pth'}, system_attrs={}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.0002, log=True, low=5e-06, step=None), 'momentum': FloatDistribution(high=0.98, log=False, low=0.88, step=None), 'epochs': IntDistribution(high=8, log=False, low=5, step=1)}, trial_id=7, value=None)


### Resultados del Experimento 4

El mejor desempeño del Experimento 4 se obtuvo en el trial 6:

- **QWK ≈ 0.309**
- learning rate ≈ 1.42e-4
- momentum ≈ 0.977
- epochs = 7
- batch size = 16

Este resultado fue muy cercano al mejor desempeño del Experimento 3, aunque no lo superó.

### Experimentos de mejores hiperparámetros sin optuna

### Experimento manual 1 — Solo FC con hiperparámetros del Experimento 3

Se utilizan los mejores hiperparámetros encontrados en el Experimento 3:

- learning rate ≈ 4.1e-4  
- momentum ≈ 0.83  
- epochs = 8  
- batch size = 16  

Pero **sin fine-tuning de layer4**.

Este experimento permite evaluar si el desempeño del Experimento 3 se debe principalmente a los hiperparámetros o al ajuste de capas profundas.

In [65]:
EXP_NAME = "exp_manual_fc_only_best_exp3"

BATCH_SIZE_EXP7 = 16

train_ds7, val_ds7, train_dl7, val_dl7 = create_dataloaders(
    new_train_directory,
    new_val_directory,
    BATCH_SIZE_EXP7,
    NUM_WORKERS
)

EXP7_LR = 4.1e-4
EXP7_MOMENTUM = 0.83
EXP7_EPOCHS = 8

model = build_resnet50_model(
    num_classes=5,
    unfreeze_layer4=False,
    unfreeze_layer3=False
)

criterion = nn.CrossEntropyLoss()

model, kappa, history, model_path, cm_path = train_val(
    model,
    criterion,
    {"train": train_dl7, "val": val_dl7},
    {"train": train_ds7, "val": val_ds7},
    device,
    EXP7_EPOCHS,
    EXP7_LR,
    EXP7_MOMENTUM,
    EXP_NAME
)

results.append({
    "experiment": EXP_NAME,
    "trial": "manual_best_exp3_params",
    "batch_size": BATCH_SIZE_EXP7,
    "unfreeze_layer4": False,
    "unfreeze_layer3": False,
    "lr": EXP7_LR,
    "momentum": EXP7_MOMENTUM,
    "epochs": EXP7_EPOCHS,
    "kappa": kappa,
    "model_path": model_path,
    "cm_path": cm_path
})

all_histories.append(history)

pd.DataFrame(results).to_csv(f"/kaggle/working/{EXP_NAME}_results.csv", index=False)

print("Kappa exp7:", kappa)
print("Modelo:", model_path)
print("Matriz:", cm_path)

Epoch 0/7
----------


100%|██████████| 733/733 [00:43<00:00, 16.86it/s]


Train Loss: 1.4579 Acc: 32.16% Kappa: nan


100%|██████████| 184/184 [00:09<00:00, 19.36it/s]


Val Loss: 1.4246 Acc: 34.08% Kappa: 0.251

Epoch 1/7
----------


...

Val Loss: 1.4006 Acc: 34.77% Kappa: 0.270

Epoch 7/7
----------


100%|██████████| 733/733 [00:42<00:00, 17.08it/s]


Train Loss: 1.3782 Acc: 37.69% Kappa: nan


100%|██████████| 184/184 [00:09<00:00, 19.73it/s]


Val Loss: 1.3994 Acc: 35.24% Kappa: 0.282

Training complete in 7m 5s
Best val Acc: 35.28%
Best val Kappa: 0.285
Best confusion matrix saved at: /kaggle/working/confusion_matrices/exp_manual_fc_only_best_exp3_epoch_5_kappa_0.285.jpg
Modelo guardado en: /kaggle/working/models/exp_manual_fc_only_best_exp3_20260430_162222.pth
Kappa exp7: 0.28459464283230507
Modelo: /kaggle/working/models/exp_manual_fc_only_best_exp3_20260430_162222.pth
Matriz: /kaggle/working/confusion_matrices/exp_manual_fc_only_best_exp3_epoch_5_kappa_0.285.jpg


### Experimento manual 2 — Solo FC con hiperparámetros del Experimento 1

En este experimento se utilizan los mejores hiperparámetros obtenidos en el Experimento 1 (optimización sobre la capa fully connected), pero se modifica el tamaño de batch a 16.

Configuración:

- capas entrenadas: solo `fc`
- batch size = 16
- learning rate ≈ 0.0022  
- momentum ≈ 0.908  
- epochs = 4  

El objetivo es evaluar si el cambio en el tamaño de batch mejora el desempeño del modelo cuando se entrena únicamente la capa final.

In [66]:
EXP_NAME = "exp_manual_fc_only_best_exp1_batch16"

BATCH_SIZE_EXP1_MANUAL = 16

train_ds_exp1m, val_ds_exp1m, train_dl_exp1m, val_dl_exp1m = create_dataloaders(
    new_train_directory,
    new_val_directory,
    BATCH_SIZE_EXP1_MANUAL,
    NUM_WORKERS
)

EXP1_LR = 0.002198906159202721
EXP1_MOMENTUM = 0.9079534063660759
EXP1_EPOCHS = 4

model = build_resnet50_model(
    num_classes=5,
    unfreeze_layer4=False,
    unfreeze_layer3=False
)

criterion = nn.CrossEntropyLoss()

model, kappa, history, model_path, cm_path = train_val(
    model,
    criterion,
    {"train": train_dl_exp1m, "val": val_dl_exp1m},
    {"train": train_ds_exp1m, "val": val_ds_exp1m},
    device,
    EXP1_EPOCHS,
    EXP1_LR,
    EXP1_MOMENTUM,
    EXP_NAME
)

results.append({
    "experiment": EXP_NAME,
    "trial": "manual_best_exp1",
    "batch_size": BATCH_SIZE_EXP1_MANUAL,
    "unfreeze_layer4": False,
    "unfreeze_layer3": False,
    "lr": EXP1_LR,
    "momentum": EXP1_MOMENTUM,
    "epochs": EXP1_EPOCHS,
    "kappa": kappa,
    "model_path": model_path,
    "cm_path": cm_path
})

all_histories.append(history)

pd.DataFrame(results).to_csv(f"/kaggle/working/{EXP_NAME}_results.csv", index=False)

print("Kappa manual exp1:", kappa)
print("Modelo:", model_path)
print("Matriz:", cm_path)

Epoch 0/3
----------


100%|██████████| 733/733 [00:42<00:00, 17.06it/s]


Train Loss: 1.4299 Acc: 33.38% Kappa: nan


100%|██████████| 184/184 [00:09<00:00, 19.53it/s]


Val Loss: 1.4089 Acc: 35.11% Kappa: 0.280

Epoch 1/3
----------


...

Val Loss: 1.4228 Acc: 35.38% Kappa: 0.271

Epoch 3/3
----------


100%|██████████| 733/733 [00:42<00:00, 17.08it/s]


Train Loss: 1.3641 Acc: 39.15% Kappa: nan


100%|██████████| 184/184 [00:09<00:00, 19.77it/s]


Val Loss: 1.4328 Acc: 34.66% Kappa: 0.256

Training complete in 3m 32s
Best val Acc: 35.11%
Best val Kappa: 0.280
Best confusion matrix saved at: /kaggle/working/confusion_matrices/exp_manual_fc_only_best_exp1_batch16_epoch_0_kappa_0.280.jpg
Modelo guardado en: /kaggle/working/models/exp_manual_fc_only_best_exp1_batch16_20260430_162637.pth
Kappa manual exp1: 0.27982842402685393
Modelo: /kaggle/working/models/exp_manual_fc_only_best_exp1_batch16_20260430_162637.pth
Matriz: /kaggle/working/confusion_matrices/exp_manual_fc_only_best_exp1_batch16_epoch_0_kappa_0.280.jpg


### Experimento manual 3 — FC + layer4 con hiperparámetros del Experimento 3

En este experimento se reentrena manualmente la configuración más prometedora identificada por Optuna en el Experimento 3.

Configuración:

- capas entrenadas: `fc + layer4`
- batch size = 16
- learning rate ≈ 4.1e-4
- momentum ≈ 0.83
- epochs = 8

El objetivo es validar si la combinación óptima encontrada por Optuna se mantiene al repetir el entrenamiento de forma manual.

In [67]:
EXP_NAME = "exp_manual_fc_layer4_best_exp3_params_batch16"

BATCH_SIZE_EXP9 = 16

train_ds9, val_ds9, train_dl9, val_dl9 = create_dataloaders(
    new_train_directory,
    new_val_directory,
    BATCH_SIZE_EXP9,
    NUM_WORKERS
)

EXP9_LR = 4.1e-4
EXP9_MOMENTUM = 0.83
EXP9_EPOCHS = 8

model = build_resnet50_model(
    num_classes=5,
    unfreeze_layer4=True,
    unfreeze_layer3=False
)

criterion = nn.CrossEntropyLoss()

model, kappa, history, model_path, cm_path = train_val(
    model,
    criterion,
    {"train": train_dl9, "val": val_dl9},
    {"train": train_ds9, "val": val_ds9},
    device,
    EXP9_EPOCHS,
    EXP9_LR,
    EXP9_MOMENTUM,
    EXP_NAME
)

results.append({
    "experiment": EXP_NAME,
    "trial": "manual_best_exp3_params",
    "batch_size": BATCH_SIZE_EXP9,
    "unfreeze_layer4": True,
    "unfreeze_layer3": False,
    "lr": EXP9_LR,
    "momentum": EXP9_MOMENTUM,
    "epochs": EXP9_EPOCHS,
    "kappa": kappa,
    "model_path": model_path,
    "cm_path": cm_path
})

all_histories.append(history)

pd.DataFrame(results).to_csv(f"/kaggle/working/{EXP_NAME}_results.csv", index=False)

print("Kappa exp9:", kappa)
print("Modelo:", model_path)
print("Matriz:", cm_path)

Epoch 0/7
----------


100%|██████████| 733/733 [00:54<00:00, 13.56it/s]


Train Loss: 1.4588 Acc: 32.19% Kappa: nan


100%|██████████| 184/184 [00:09<00:00, 19.88it/s]


Val Loss: 1.4202 Acc: 34.15% Kappa: 0.242

Epoch 1/7
----------


...

Val Loss: 1.3952 Acc: 35.93% Kappa: 0.288

Epoch 7/7
----------


100%|██████████| 733/733 [00:53<00:00, 13.76it/s]


Train Loss: 1.3559 Acc: 39.41% Kappa: nan


100%|██████████| 184/184 [00:09<00:00, 19.87it/s]


Val Loss: 1.3945 Acc: 35.72% Kappa: 0.289

Training complete in 8m 22s
Best val Acc: 35.72%
Best val Kappa: 0.289
Best confusion matrix saved at: /kaggle/working/confusion_matrices/exp_manual_fc_layer4_best_exp3_params_batch16_epoch_7_kappa_0.289.jpg
Modelo guardado en: /kaggle/working/models/exp_manual_fc_layer4_best_exp3_params_batch16_20260430_163515.pth
Kappa exp9: 0.28945729465652004
Modelo: /kaggle/working/models/exp_manual_fc_layer4_best_exp3_params_batch16_20260430_163515.pth
Matriz: /kaggle/working/confusion_matrices/exp_manual_fc_layer4_best_exp3_params_batch16_epoch_7_kappa_0.289.jpg


### Experimento manual 4 — FC + layer4 con hiperparámetros del Experimento 1 (batch 32)

En este experimento se evalúa la combinación de:

- fine-tuning de `layer4`;
- hiperparámetros óptimos del Experimento 1;
- batch size = 32.

Configuración:

- capas entrenadas: `fc + layer4`
- batch size = 32
- learning rate ≈ 0.0022  
- momentum ≈ 0.908  
- epochs = 4  

El objetivo es analizar si los hiperparámetros que funcionaron bien en un esquema sin fine-tuning (Experimento 1) también son efectivos cuando se entrenan capas profundas.

In [68]:
EXP_NAME = "exp_manual_fc_layer4_best_exp1_params_batch32"

BATCH_SIZE_EXP10 = 32

train_ds10, val_ds10, train_dl10, val_dl10 = create_dataloaders(
    new_train_directory,
    new_val_directory,
    BATCH_SIZE_EXP10,
    NUM_WORKERS
)

EXP10_LR = 0.002198906159202721
EXP10_MOMENTUM = 0.9079534063660759
EXP10_EPOCHS = 4

model = build_resnet50_model(
    num_classes=5,
    unfreeze_layer4=True,
    unfreeze_layer3=False
)

criterion = nn.CrossEntropyLoss()

model, kappa, history, model_path, cm_path = train_val(
    model,
    criterion,
    {"train": train_dl10, "val": val_dl10},
    {"train": train_ds10, "val": val_ds10},
    device,
    EXP10_EPOCHS,
    EXP10_LR,
    EXP10_MOMENTUM,
    EXP_NAME
)

results.append({
    "experiment": EXP_NAME,
    "trial": "manual_best_exp1_params",
    "batch_size": BATCH_SIZE_EXP10,
    "unfreeze_layer4": True,
    "unfreeze_layer3": False,
    "lr": EXP10_LR,
    "momentum": EXP10_MOMENTUM,
    "epochs": EXP10_EPOCHS,
    "kappa": kappa,
    "model_path": model_path,
    "cm_path": cm_path
})

all_histories.append(history)

pd.DataFrame(results).to_csv(f"/kaggle/working/{EXP_NAME}_results.csv", index=False)

print("Kappa exp10:", kappa)
print("Modelo:", model_path)
print("Matriz:", cm_path)

Epoch 0/3
----------


100%|██████████| 367/367 [00:51<00:00,  7.11it/s]


Train Loss: 1.4244 Acc: 34.12% Kappa: nan


100%|██████████| 92/92 [00:09<00:00, 10.07it/s]


Val Loss: 1.4115 Acc: 34.05% Kappa: 0.245

Epoch 1/3
----------


...

Val Loss: 1.4125 Acc: 34.15% Kappa: 0.264

Epoch 3/3
----------


100%|██████████| 367/367 [00:51<00:00,  7.19it/s]


Train Loss: 1.3096 Acc: 42.58% Kappa: nan


100%|██████████| 92/92 [00:09<00:00,  9.96it/s]


Val Loss: 1.4233 Acc: 36.40% Kappa: 0.301

Training complete in 4m 3s
Best val Acc: 36.40%
Best val Kappa: 0.301
Best confusion matrix saved at: /kaggle/working/confusion_matrices/exp_manual_fc_layer4_best_exp1_params_batch32_epoch_3_kappa_0.301.jpg
Modelo guardado en: /kaggle/working/models/exp_manual_fc_layer4_best_exp1_params_batch32_20260430_163922.pth
Kappa exp10: 0.3005762337678446
Modelo: /kaggle/working/models/exp_manual_fc_layer4_best_exp1_params_batch32_20260430_163922.pth
Matriz: /kaggle/working/confusion_matrices/exp_manual_fc_layer4_best_exp1_params_batch32_epoch_3_kappa_0.301.jpg
